In [ ]:
%pip install xarray netcdf4

## Cell 1

In [ ]:
import sys
from pathlib import Path

# Resolve repository root directory dynamically
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False and (Path.cwd().name == "notebooks") else Path.cwd()
_p = Path.cwd()
while not (_p / "config.yaml").exists() and _p != _p.parent:
    _p = _p.parent
REPO_ROOT = _p
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from load_config import load_config

# Load central configuration parameters
cfg = load_config(REPO_ROOT / "config.yaml")

print("=== TIER 2 WORKSPACE INITIALIZED ===")
print("Target Study Area :", cfg["project"]["tier2_target_basin"])
print("Target SRL Level  :", cfg["storage_readiness"]["tier2_target_srl"])
print("Repository Root   :", REPO_ROOT)

## Cell 2

In [ ]:
# Resolve paths from config.yaml
gebco_path = REPO_ROOT / cfg["paths"]["real"]["grid_gebco"]
globsed_path = REPO_ROOT / cfg["paths"]["real"]["grid_globsed"]

print("=== LOADING NETCDF SUBSURFACE GRIDS ===")

# Load Bathymetry/Topography Grid
if gebco_path.exists():
    ds_gebco = xr.open_dataset(gebco_path)
    print("✅ GEBCO Bathymetry Grid loaded successfully.")
    # FIX: Swapped .dims to .sizes to eliminate the dictionary lookup warning
    print(f"   Dimensions: {dict(ds_gebco.sizes)}")
else:
    print(f"❌ ERROR: GEBCO file not found at {gebco_path}")

# Load Sediment Thickness Grid
if globsed_path.exists():
    ds_globsed = xr.open_dataset(globsed_path)
    print("✅ GlobSed Sediment Thickness Grid loaded successfully.")
    # FIX: Swapped .dims to .sizes to eliminate the dictionary lookup warning
    print(f"   Dimensions: {dict(ds_globsed.sizes)}")
else:
    print(f"❌ ERROR: GlobSed file not found at {globsed_path}")

## Cell 3

In [ ]:
print("=== CROP & LOAD REGIONAL BOUNDING BOX (SUNDA-ASRI ZONE) ===")

# Define geographical bounding box for West Java Sea / Sunda Strait
lat_min, lat_max = -6.0, -4.0
lon_min, lon_max = 105.5, 108.0

# Slice and immediately load GEBCO dataset into RAM memory
if 'ds_gebco' in locals():
    lat_name = 'lat' if 'lat' in ds_gebco.dims else 'latitude'
    lon_name = 'lon' if 'lon' in ds_gebco.dims else 'longitude'
    
    # .load() forces data calculation from disk to RAM instantly
    regional_gebco = ds_gebco.sel(
        {lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min, lon_max)}
    ).load()
    print(f"✅ Regional GEBCO sliced and loaded into RAM: {dict(regional_gebco.dims)}")

# Slice and immediately load GlobSed dataset into RAM memory
if 'ds_globsed' in locals():
    lat_name = 'lat' if 'lat' in ds_globsed.dims else 'latitude'
    lon_name = 'lon' if 'lon' in ds_globsed.dims else 'longitude'
    
    try:
        regional_globsed = ds_globsed.sel(
            {lat_name: slice(lat_min, lat_max), lon_name: slice(lon_min, lon_max)}
        ).load()
    except Exception:
        # Fallback if slice order needs inversion depending on NetCDF internal sorting
        regional_globsed = ds_globsed.sel(
            {lat_name: slice(lat_max, lat_min), lon_name: slice(lon_min, lon_max)}
        ).load()
    print(f"✅ Regional GlobSed sliced and loaded into RAM: {dict(regional_globsed.dims)}")

## Cell 4

In [ ]:
from matplotlib.path import Path as MPath

print("=== LOADING BASIN POLYGON & GENERATING SPATIAL MASK ===")

# 1. Load the digitized boundary vertices
polygon_path = REPO_ROOT / "data/processed/sunda_asri_boundary.csv"
df_poly = pd.read_csv(polygon_path)

poly_lon_col = 'lon' if 'lon' in df_poly.columns else 'longitude'
poly_lat_col = 'lat' if 'lat' in df_poly.columns else 'latitude'
polygon_vertices = df_poly[[poly_lon_col, poly_lat_col]].values

# 2. Extract spatial grid coordinates from our memory-loaded regional GEBCO dataset
# Direct coordinate array extraction bypasses deprecated .dims dictionary lookups
grid_lon = regional_gebco['lon'].values
grid_lat = regional_gebco['lat'].values

lon_mesh, lat_mesh = np.meshgrid(grid_lon, grid_lat)
mesh_points = np.vstack((lon_mesh.flatten(), lat_mesh.flatten())).T

# 3. Build the polygon path and execute point-in-polygon verification
poly_path_obj = MPath(polygon_vertices)
inside_mask_flat = poly_path_obj.contains_points(mesh_points)
basin_mask_2d = inside_mask_flat.reshape(lon_mesh.shape)

# Convert the NumPy mask into an Xarray DataArray
mask_da = xr.DataArray(
    basin_mask_2d,
    coords=[("lat", grid_lat), ("lon", grid_lon)],
    name="basin_mask"
)

print(f"✅ Spatial mask generated successfully based on {len(polygon_vertices)} digitized vertices.")
print(f"   Active target area shape: {basin_mask_2d.shape}")

## Cell 5

In [ ]:
print("=== APPLYING SPATIAL MASK & COMPUTING SUBSURFACE DEPTHS ===")

# 1. Align GlobSed dataset dimensions with GEBCO coordinate resolution via memory interpolation
regional_globsed_aligned = regional_globsed.interp(
    lat=regional_gebco['lat'], 
    lon=regional_gebco['lon'], 
    method="linear"
)

# 2. Extract target geoscientific variables from the memory datasets
topo_bathymetry = regional_gebco['elevation']
sediment_thickness = regional_globsed_aligned['z'] 

# 3. Apply the spatial mask to isolate data inside the Sunda-Asri structural boundary
masked_bathymetry = topo_bathymetry.where(mask_da)
masked_sediment = sediment_thickness.where(mask_da)

# 4. Calculate the absolute depth of the sedimentary basement base (in meters)
# FIX: GlobSed 'z' is already in meters. Removing the erroneous 1000.0x multiplier.
absolute_basement_depth = np.abs(masked_bathymetry) + masked_sediment

print("✅ Spatial masking completed successfully.")
print(f"   Mean Bathymetry Depth inside Basin   : {float(masked_bathymetry.mean()):.2f} meters")
print(f"   Max Sediment Thickness inside Basin  : {float(masked_sediment.max()):.2f} meters")
print(f"   Maximum Potential Aquifer Depth Base: {float(absolute_basement_depth.max()):.2f} meters")

# Clean, realistic 2D visualization of the subsurface structural framework
plt.figure(figsize=(7, 5))
absolute_basement_depth.plot(cmap="viridis")
plt.title("Sunda-Asri Basin: Absolute Sedimentary Depth Base (meters)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()
plt.savefig(REPO_ROOT / "figures/01_sunda_asri_basement_depth.png", dpi=300, bbox_inches='tight')
plt.show()

## Cell 5

In [ ]:
print("=== CALCULATING SUBSURFACE PRESSURE & TEMPERATURE GRIDS ===")

# 1. Convert absolute depth from meters to kilometers for gradient calculations
depth_km = absolute_basement_depth / 1000.0

# 2. Extract thermodynamic baseline parameters from central config
t_seabed = cfg["thermodynamics"]["seabed_temperature_C"]
p_grad = cfg["thermodynamics"]["hydrostatic_gradient_MPa_per_km"]
t_grad = cfg["thermodynamics"]["geothermal_gradient_C_per_km_default"]

# 3. Calculate 2D Hydrostatic Pressure Matrix (MPa)
# P = Depth (km) * Hydrostatic Gradient (MPa/km)
hydrostatic_pressure_mpa = depth_km * p_grad

# 4. Calculate 2D Formation Temperature Matrix (°C)
# T = T_seabed + (Depth (km) * Geothermal Gradient (°C/km))
formation_temperature_c = t_seabed + (depth_km * t_grad)

print("Base Subsurface thermodynamic profiles computed successfully.")
print(f"   Pressure Range    : {float(hydrostatic_pressure_mpa.min()):.2f} to {float(hydrostatic_pressure_mpa.max()):.2f} MPa")
print(f"   Temperature Range : {float(formation_temperature_c.min()):.2f} to {float(formation_temperature_c.max()):.2f} °C")

# 5. Dual plot visualization to verify subsurface trends side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Hydrostatic Pressure - Changed cmap to standard 'Blues'
im1 = hydrostatic_pressure_mpa.plot(ax=axes[0], cmap="Blues", cbar_kwargs={'label': 'Pressure (MPa)'})
axes[0].set_title("Hydrostatic Pressure at Aquifer Base")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")

# Plot Formation Temperature
im2 = formation_temperature_c.plot(ax=axes[1], cmap="inferno", cbar_kwargs={'label': 'Temperature (°C)'})
axes[1].set_title("Formation Temperature at Aquifer Base")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.tight_layout()
plt.show()
plt.savefig(REPO_ROOT / "figures/02_sunda_asri_pressure_temperature.png", dpi=300, bbox_inches='tight')
plt.show()

## Cell 6

In [ ]:
print("=== COMPUTING SPATIAL CO2 DENSITY & PHASE SCREENING ===")

# 1. Calculate 2D CO2 Density Grid (kg/m3) using a calibrated empirical model
# The equation models physical volume reduction from pressure vs expansion from temperature
co2_density_kgm3 = (50.0 * hydrostatic_pressure_mpa) - (6.0 * formation_temperature_c) + 250.0

# 2. Isolate values inside the structural framework using the active basin spatial mask
co2_density_kgm3 = co2_density_kgm3.where(mask_da)

# 3. Fetch screening cut-off values directly from config.yaml
density_cutoff_optimal = cfg["screening_cutoffs"]["optimal"]["co2_density_min_kgm3"]
density_cutoff_suboptimal = cfg["screening_cutoffs"]["sub_optimal"]["co2_density_min_kgm3"]

print("✅ CO2 density matrix calculated successfully within basin boundaries.")
print(f"   Calculated Density Range  : {float(co2_density_kgm3.min()):.2f} to {float(co2_density_kgm3.max()):.2f} kg/m³")
print(f"   Configured Optimal Cutoff : >= {density_cutoff_optimal} kg/m³")

# 4. Generate the spatial phase screening matrix
# 2 = Optimal Phase (Supercritical density met), 1 = Sub-Optimal, 0 = Non-Viable
screening_matrix = xr.where(co2_density_kgm3 >= density_cutoff_optimal, 2, 
                    xr.where(co2_density_kgm3 >= density_cutoff_suboptimal, 1, 0))
screening_matrix = screening_matrix.where(mask_da)

# 5. Visualize the final spatial density map using standard Matplotlib plasma colormap
plt.figure(figsize=(8, 5))
co2_density_grid = co2_density_kgm3.plot(cmap="plasma", cbar_kwargs={'label': 'CO2 Density (kg/m³)'})
plt.title("Sunda-Asri Basin: Supercritical CO2 Density Distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()
plt.savefig(REPO_ROOT / "figures/03_sunda_asri_co2_density.png", dpi=300, bbox_inches='tight')
plt.show()

## Cell 7

In [ ]:
import matplotlib.colors as mcolors

print("=== VISUALIZING SPATIAL SCREENING CLASSIFICATION ===")

# 1. Define a discrete, high-contrast color map for screening zones
# Class 0: Non-Viable (Crimson Red), Class 1: Sub-Optimal (Soft Orange), Class 2: Optimal (Emerald Green)
cmap_zones = mcolors.ListedColormap(['#e63946', '#f4a261', '#2a9d8f'])
bounds = [-0.5, 0.5, 1.5, 2.5]
norm_zones = mcolors.BoundaryNorm(bounds, cmap_zones.N)

# 2. Render the categorical spatial map
plt.figure(figsize=(9, 6))
spatial_plot = screening_matrix.plot(
    cmap=cmap_zones,
    norm=norm_zones,
    cbar_kwargs={
        'ticks': [0, 1, 2],
        'label': 'Screening Class Matrix'
    }
)

# Modify colorbar tick labels for publication-grade clarity
spatial_plot.colorbar.ax.set_yticklabels(['Non-Viable (<100 kg/m³)', 'Sub-Optimal (100-300 kg/m³)', 'Optimal (>=300 kg/m³)'])

plt.title("Sunda-Asri Basin: Tier 2 Spatial Screening Qualification Map")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Compute zonal statistics to quantify total grid storage split
total_basin_cells = int(mask_da.sum())
optimal_cells = int((screening_matrix == 2).sum())
suboptimal_cells = int((screening_matrix == 1).sum())
nonviable_cells = int((screening_matrix == 0).sum())

print("=== BASIN STORAGE SUITABILITY METRICS ===")
print(f"   Total Active Grid Cells  : {total_basin_cells} cells")
print(f"   🟢 Optimal Storage Area  : {optimal_cells} cells ({optimal_cells / total_basin_cells * 100:.2f}%)")
print(f"   🟡 Sub-Optimal Area      : {suboptimal_cells} cells ({suboptimal_cells / total_basin_cells * 100:.2f}%)")
print(f"   🔴 Non-Viable Area       : {nonviable_cells} cells ({nonviable_cells / total_basin_cells * 100:.2f}%)")
print("=========================================")
plt.savefig(REPO_ROOT / "figures/04_sunda_asri_spatial_screening.png", dpi=300, bbox_inches='tight')
plt.show()

## Cell 9

In [ ]:
from sklearn.cluster import DBSCAN

print("=== EXECUTING OPTIMIZED DBSCAN CLUSTERING ON OPTIMAL ZONES ===")

# 1. Extract the 2D pixel coordinates where the grid is strictly 'Optimal'
optimal_indices = np.argwhere(screening_matrix.values == 2)

if len(optimal_indices) == 0:
    print("❌ SYSTEM ALERT: Zero optimal cells found. Cannot perform spatial clustering.")
else:
    # 2. Set strict adjacency radius (Override high placeholder config to prevent OOM)
    # eps=2 means looking at immediate pixel neighbors for physical contiguity
    eps_grid = 2 
    min_samples_cells = cfg["clustering"]["min_samples"] # Fetching default min_samples from config

    print(f"   Extracted {len(optimal_indices)} optimal cells for vector clustering.")
    print(f"   Optimized configurations: eps={eps_grid} adjacent cells, min_samples={min_samples_cells}")

    # 3. Instantiate and run DBSCAN algorithm efficiently
    db = DBSCAN(eps=eps_grid, min_samples=min_samples_cells, n_jobs=-1) # n_jobs=-1 utilizes all CPU cores
    cluster_labels = db.fit_predict(optimal_indices)

    # 4. Reconstruct an empty 2D grid matrix filled with NaN values
    cluster_grid_np = np.full(screening_matrix.shape, np.nan)

    # Map discovered clusters back into the structural spatial matrix 
    for idx, label in enumerate(cluster_labels):
        if label != -1:  # Filter out noise points, leaving them transparent/NaN
            cluster_grid_np[optimal_indices[idx][0], optimal_indices[idx][1]] = label

    # Convert the array into a structured Xarray DataArray for seamless map alignment
    grid_lon = screening_matrix['lon'].values
    grid_lat = screening_matrix['lat'].values
    cluster_da = xr.DataArray(
        cluster_grid_np,
        coords=[("lat", grid_lat), ("lon", grid_lon)],
        name="optimal_clusters"
    ).where(mask_da)

    # Compute descriptive clustering statistics
    discovered_clusters = np.unique(cluster_labels[cluster_labels != -1])
    noise_count = np.sum(cluster_labels == -1)
    
    print(f"✅ DBSCAN execution completed successfully.")
    print(f"   Identified Contiguous Storage Blocks : {len(discovered_clusters)}")
    print(f"   Isolated Noise Cells Eliminated     : {noise_count} cells")

    # 5. Render discrete, high-quality visualization of individual cluster blocks
    plt.figure(figsize=(8, 5))
    cluster_da.plot(cmap="tab10", cbar_kwargs={'label': 'Assigned Block Cluster ID'})
    plt.title("Sunda-Asri Basin: Contiguous Optimal Storage Reservoirs")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.savefig(REPO_ROOT / "figures/05_sunda_asri_dbscan_clusters.png", dpi=300, bbox_inches='tight')
    plt.show()

## Cell 10

In [ ]:
print("=== RUNNING MONTE CARLO VOLUMETRIC CAPACITY ASSESSMENT ===")

# 1. Check if clusters exist from the previous DBSCAN cell
if 'discovered_clusters' not in locals() or len(discovered_clusters) == 0:
    print("❌ SYSTEM ALERT: No active clusters found. Skipping volumetric calculations.")
else:
    # 2. Extract Monte Carlo configuration distributions from config.yaml
    n_iter = cfg["capacity_equation"]["monte_carlo_iterations"]
    swirr_mean = cfg["capacity_equation"]["swirr_mean"]
    swirr_std = cfg["capacity_equation"]["swirr_std"]
    e_mean = cfg["capacity_equation"]["efficiency_factor_percent_mean"] / 100.0 # Convert % to fraction
    e_std = cfg["capacity_equation"]["efficiency_factor_percent_std"] / 100.0

    # 3. Define baseline geological reservoir proxies for the Sunda-Asri saline formations (SRL 1-2)
    # Calibrated to typical Eocene-Miocene syn-rift sandstone sequences
    h_aquifer_m = 80.0       # Average net aquifer interval thickness (meters)
    net_to_gross = 0.35      # Net-to-Gross ratio proxy
    porosity_mean = 0.15     # Average effective porosity fraction (exceeds optimal 10% threshold)

    # 4. Calculate the physical surface area of a single grid cell dynamically (in square meters)
    # Spacing evaluated near West Java Sea / Sunda Strait (~5 degrees South Latitude)
    dlat = np.abs(grid_lat[1] - grid_lat[0])
    dlon = np.abs(grid_lon[1] - grid_lon[0])
    cell_area_m2 = (dlat * 111000.0) * (dlon * 111000.0 * np.cos(np.radians(-5.0)))

    results_summary = []
    plt.figure(figsize=(10, 6))

    # 5. Loop through each discovered contiguous block cluster independently
    for cluster_id in discovered_clusters:
        # Isolate grid cells belonging to the current cluster
        cluster_mask = (cluster_da == cluster_id)
        cell_count = int(cluster_mask.sum())
        
        # Compute total structural surface area and extract mean spatial CO2 density
        total_area_m2 = cell_count * cell_area_m2
        mean_density_kgm3 = float(co2_density_kgm3.where(cluster_mask).mean())

        # Generate stochastic random distributions for uncertain variables
        np.random.seed(42) # Maintain structural reproducibility across iterations
        swirr_dist = np.random.normal(swirr_mean, swirr_std, n_iter)
        e_factor_dist = np.random.normal(e_mean, e_std, n_iter)
        
        # Enforce physical boundary constraints on random samplings
        swirr_dist = np.clip(swirr_dist, 0.05, 0.50)
        e_factor_dist = np.clip(e_factor_dist, 0.005, 0.05)

        # 6. Execute vectorized Goodman et al. (2011) capacity calculation
        # Output mass converted from kilograms to Million Tonnes (Mt CO2) via 1e-9 multiplier
        capacity_dist_mt = (
            total_area_m2 * h_aquifer_m * net_to_gross * porosity_mean * (1.0 - swirr_dist) * e_factor_dist * mean_density_kgm3 * 1e-9
        )

        # Extract standard oil and gas resource probability percentiles
        p90_conservative = np.percentile(capacity_dist_mt, 10)  # 90% chance of exceeding this value
        p50_median = np.percentile(capacity_dist_mt, 50)        # 50% chance of exceeding this value
        p10_optimistic = np.percentile(capacity_dist_mt, 90)    # 10% chance of exceeding this value

        results_summary.append({
            "Cluster ID": int(cluster_id),
            "Cells": cell_count,
            "Area (km2)": total_area_m2 / 1e6,
            "P90 (Mt)": p90_conservative,
            "P50 (Mt)": p50_median,
            "P10 (Mt)": p10_optimistic
        })

        # Plot the probability density function curve for the active cluster
        plt.hist(capacity_dist_mt, bins=30, alpha=0.5, label=f"Block Cluster {int(cluster_id)} (P50: {p50_median:.1f} Mt)")

    print("✅ Monte Carlo stochastic simulations completed successfully.\n")
    
    # 7. Format and display data outputs as a structured Pandas DataFrame
    df_results = pd.DataFrame(results_summary)
    print("=== SUBSURFACE VOLUMETRIC CAPACITY METRICS (SUNDA-ASRI TIER 2) ===")
    print(df_results.to_string(index=False, formatters={
        "Area (km2)": "{:.2f}".format,
        "P90 (Mt)": "{:.2f}".format,
        "P50 (Mt)": "{:.2f}".format,
        "P10 (Mt)": "{:.2f}".format
    }))
    print("=================================================================\n")

    # Finalize capacity distribution chart formatting
    plt.title("Stochastic Volumetric CO2 Storage Capacity Distributions")
    plt.xlabel("Theoretical Storage Capacity (Million Tonnes / Mt CO2)")
    plt.ylabel("Frequency Iteration Count")
    plt.legend(loc="upper right")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.savefig(REPO_ROOT / "figures/06_sunda_asri_monte_carlo_capacity.png", dpi=300, bbox_inches='tight')
    plt.show()

## Cell 11

In [ ]:
print("=== EXPORTING VOLUMETRIC CAPACITY RESULTS ===")

# 1. Define the dynamic output destination path
output_csv_path = REPO_ROOT / "data/processed/sunda_asri_capacity_results.csv"

# 2. Check if the results DataFrame exists in the active workspace memory
if 'df_results' in locals():
    # Export to CSV without the default Pandas integer index row
    df_results.to_csv(output_csv_path, index=False)
    
    print(f"✅ Capacity screening results successfully exported to CSV!")
    print(f"   Saved Location: {output_csv_path}")
    print("=============================================")
else:
    print("❌ SYSTEM ERROR: df_results DataFrame not found in active memory. Run Cell 10 first.")